# slv_pedidos
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · entidad Pedidos (cabecera) |
| Propósito | bronze.orders_raw → silver.pedidos (SCD1, estado actual del pedido). |
| Integridad | total frente a suma de ítems y pedidos sin ítems → slv_controles_integridad.py |
| Reglas | reglas_calidad.REGLAS["pedidos"] |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.2 · 2026-10-21 · el descarte pasa a la vista validado_pedidos: limpio_ ya no descarta, así la cuarentena recibe las filas inválidas |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun import TECNICAS, con_secuencia, leer_bronze, parametros_cdc
from reglas_calidad import reglas

# AUTO CDC: nombre actual de la API; apply_changes es el nombre anterior.
aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Vista de limpieza `limpio_pedidos`
Lee los cambios de bronze **como stream** (solo lo nuevo en cada corrida) y:
1. agrega el **orden de los cambios** (`_secuencia`: fecha efectiva, `_lsn`, `_seqval`);
2. **renombra** a snake_case en español y **normaliza** valores;
3. aplica las reglas `fallar` (detiene el pipeline) y `advertir` (entra y queda la métrica).

Aquí **no se descarta nada**: esta vista la leen dos rutas, la validación (sección 3) y la cuarentena (`slv_cuarentena`). Un `expect_all_or_drop` en una vista se aplica a todo el que la lee, así que si descartara aquí, la cuarentena nunca vería las filas inválidas.

In [ ]:
@dlt.view(name="limpio_pedidos", comment="Pedidos de bronze renombrados y limpios (sin descarte).")
@dlt.expect_all_or_fail(reglas("pedidos", "fallar"))      # rompe el modelo → detiene el pipeline
@dlt.expect_all(reglas("pedidos", "advertir"))            # mejorable → entra y queda la métrica
def limpio_pedidos():
    df = con_secuencia(leer_bronze("orders_raw"))
    return df.select(
        F.col("OrderID").alias("id_pedido"),
        F.col("CustomerID").alias("id_cliente"),
        F.col("OrderDate").alias("fecha_pedido"),
        F.lower("Channel").alias("canal"),
        F.lower("Status").alias("estado"),
        F.col("TotalAmount").alias("monto_total"),
        F.col("CreatedAt").alias("creado_en"),
        F.col("UpdatedAt").alias("actualizado_en"),
        *TECNICAS)

## 3. Vista de validación `validado_pedidos`
La puerta de entrada a silver: aplica las reglas `descartar` y solo deja pasar las filas que las cumplen. Lo que descarta lo recoge `slv_cuarentena` desde `limpio_pedidos`, con las **mismas reglas**: cada fila termina en silver o en la cuarentena, nunca en las dos ni en ninguna.

In [ ]:
@dlt.view(name="validado_pedidos", comment="Pedidos que cumplen las reglas de descarte: lo que entra a silver.")
@dlt.expect_all_or_drop(reglas("pedidos", "descartar"))   # inválido → no entra (lo recoge la cuarentena)
def validado_pedidos():
    return dlt.read_stream("limpio_pedidos")

## 4. Tabla(s) silver con AUTO CDC: `pedidos`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD1** → estado actual (una fila por clave).

In [ ]:
dlt.create_streaming_table(name="pedidos", comment="Pedidos: estado actual (SCD1) desde CDC de Azure SQL.")
aplicar_cdc(target="pedidos", source="validado_pedidos", **parametros_cdc("id_pedido"))
# Lo que no cumple una regla de descarte lo recoge slv_cuarentena.py.